# Multi-ML-Framework Sample Training

This notebook demonstrates the orchestration shape for training different model families on the same Quant Warehouse dataset. The universe is MAG7 and the data vendors are `yfinance` and `fmp`; the ML frameworks are RAPIDS cuML RandomForest, PyTorch, and FlairNLP with a tiny pretrained transformer.

All market data, numeric features, and optimal-trade targets come from Quant Warehouse. The notebook does not write temporary CSV or parquet data files.

In [1]:
from __future__ import annotations

from pathlib import Path
from time import perf_counter
import math
import random
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "quant_orchestrator").exists():
            return candidate
    raise RuntimeError(f"Could not find quant-orchestrator repo root from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
for path in (REPO_ROOT, REPO_ROOT.parent / "quant-warehouse"):
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

warnings.filterwarnings("ignore", category=UserWarning)

from quant_warehouse import Warehouse
from quant_warehouse.target_engineering import build_label_panel

from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, roc_auc_score
from sklearn.preprocessing import StandardScaler

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

import flair
from quant_orchestrator.platforms.ml_frameworks.flair.shared import (
    predict_classification_regression,
    train_classification_regression_multitask,
)


In [2]:
SYMBOLS = ["AAPL", "MSFT", "NVDA", "AMZN", "META", "GOOGL", "TSLA"]
PROVIDERS = ("yfinance", "fmp")
START = "2020-01-01"
END = None
OHLCV_COLUMNS = ["open", "high", "low", "close", "volume"]
LABEL_K_PARAMS = {"YE": list(range(1, 13))}
TRAIN_END = pd.Timestamp("2024-12-31")
DEV_END = pd.Timestamp("2025-12-31")
RANDOM_SEED = 20260629
TORCH_EPOCHS = 20
FLAIR_EPOCHS = 1
FLAIR_TRANSFORMER = "prajjwal1/bert-tiny"
ARTIFACT_DIR = REPO_ROOT / "artifacts" / "notebooks" / "mult-ml-frameworks"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

TORCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
flair.device = TORCH_DEVICE

runtime_config = pd.DataFrame(
    [
        {"setting": "symbols", "value": ", ".join(SYMBOLS)},
        {"setting": "providers", "value": ", ".join(PROVIDERS)},
        {"setting": "date_range", "value": f"{START} to latest warehouse row"},
        {"setting": "target_engineering", "value": f"optimal_trading {LABEL_K_PARAMS}"},
        {"setting": "torch_cuda_available", "value": torch.cuda.is_available()},
        {"setting": "torch_device", "value": str(TORCH_DEVICE)},
        {"setting": "flair_transformer", "value": FLAIR_TRANSFORMER},
    ]
)
display(runtime_config)

,setting,value
0,symbols,"AAPL, MSFT, NVDA, AMZN, META, GOOGL, TSLA"
1,providers,"yfinance, fmp"
2,date_range,2020-01-01 to latest warehouse row
3,target_engineering,"optimal_trading {'YE': [1, 2, 3, 4, 5, 6, 7, 8..."
4,torch_cuda_available,True
5,torch_device,cuda
6,flair_transformer,prajjwal1/bert-tiny


## Dataset Construction

Each provider gets an independent MAG7 dataset. The numeric feature matrix is adjusted OHLCV from Quant Warehouse. The binary label and return percentile target are generated by Quant Warehouse target engineering with one optimal-trading label configuration: `YE: 1..12`.

In [3]:
def normalize_price_frame(prices: pd.DataFrame) -> pd.DataFrame:
    frame = prices.copy()
    frame.columns = [str(col).lower() for col in frame.columns]
    frame.index = pd.to_datetime(frame.index).tz_localize(None)
    frame = frame.sort_index()
    missing = [col for col in OHLCV_COLUMNS if col not in frame.columns]
    if missing:
        raise ValueError(f"Missing OHLCV columns from warehouse prices: {missing}")
    return frame[OHLCV_COLUMNS].astype(float).dropna()


def row_to_key_value_text(row: pd.Series) -> str:
    return " ".join(
        [
            f"symbol={row['symbol']}",
            f"provider={row['provider']}",
            f"datetime={pd.Timestamp(row['date']).date()}",
            f"event={row.get('event', 'label')}",
            f"horizon={row.get('horizon', 'unknown')}",
            f"open={row['open']:.6f}",
            f"high={row['high']:.6f}",
            f"low={row['low']:.6f}",
            f"close={row['close']:.6f}",
            f"volume={row['volume']:.0f}",
        ]
    )



def load_provider_dataset(provider: str) -> pd.DataFrame:
    provider_frames = []
    for symbol in SYMBOLS:
        prices = normalize_price_frame(Warehouse().read_prices(symbol, provider=provider, start=START, end=END))
        labels = build_label_panel(
            {symbol: prices.copy()},
            k_params=LABEL_K_PARAMS,
            solver_mode="period_top_k",
            add_rank_labels=True,
            deduplicate=True,
            max_workers=1,
        ).reset_index()
        labels["date"] = pd.to_datetime(labels["date"]).dt.tz_localize(None)
        labels["symbol"] = labels["symbol"].astype(str)
        labels = labels.dropna(subset=["date", "symbol", "target", "rank_y"]).copy()
        labels["target"] = labels["target"].astype(int)
        labels["rank_y"] = labels["rank_y"].astype(float)

        symbol_prices = prices.reset_index().rename(columns={"index": "date"})
        symbol_prices["symbol"] = symbol
        symbol_dataset = labels.merge(symbol_prices, on=["date", "symbol"], how="inner")
        symbol_dataset["provider"] = provider
        provider_frames.append(symbol_dataset)

    dataset = pd.concat(provider_frames, ignore_index=True).sort_values(
        ["symbol", "date", "event", "horizon"]
    )
    dataset["text"] = dataset.apply(row_to_key_value_text, axis=1)
    dataset = dataset.reset_index(drop=True)
    if dataset["target"].nunique() < 2:
        raise ValueError(f"{provider} dataset has only one class after joining features and labels")
    return dataset


def chronological_split(dataset: pd.DataFrame) -> dict[str, pd.DataFrame]:
    train = dataset[dataset["date"] <= TRAIN_END].copy()
    dev = dataset[(dataset["date"] > TRAIN_END) & (dataset["date"] <= DEV_END)].copy()
    test = dataset[dataset["date"] > DEV_END].copy()
    if min(len(train), len(dev), len(test)) == 0:
        ordered = dataset.sort_values("date").reset_index(drop=True)
        first = max(2, int(len(ordered) * 0.70))
        second = max(first + 1, int(len(ordered) * 0.85))
        train = ordered.iloc[:first].copy()
        dev = ordered.iloc[first:second].copy()
        test = ordered.iloc[second:].copy()
    return {"train": train, "dev": dev, "test": test}


def scaled_feature_splits(splits: dict[str, pd.DataFrame]) -> tuple[dict[str, np.ndarray], StandardScaler]:
    scaler = StandardScaler()
    arrays = {
        "train": scaler.fit_transform(splits["train"][OHLCV_COLUMNS]).astype("float32"),
        "dev": scaler.transform(splits["dev"][OHLCV_COLUMNS]).astype("float32"),
        "test": scaler.transform(splits["test"][OHLCV_COLUMNS]).astype("float32"),
    }
    return arrays, scaler

In [4]:
provider_datasets = {}
provider_splits = {}
provider_arrays = {}
provider_scalers = {}
audit_rows = []

for provider in PROVIDERS:
    dataset = load_provider_dataset(provider)
    splits = chronological_split(dataset)
    arrays, scaler = scaled_feature_splits(splits)
    provider_datasets[provider] = dataset
    provider_splits[provider] = splits
    provider_arrays[provider] = arrays
    provider_scalers[provider] = scaler
    audit_rows.append(
        {
            "provider": provider,
            "symbols": dataset["symbol"].nunique(),
            "rows": len(dataset),
            "train_rows": len(splits["train"]),
            "dev_rows": len(splits["dev"]),
            "test_rows": len(splits["test"]),
            "positive_rate": dataset["target"].mean(),
            "first_label_date": dataset["date"].min().date(),
            "last_label_date": dataset["date"].max().date(),
            "unique_horizons": dataset["horizon"].nunique(),
        }
    )

summary = pd.DataFrame(audit_rows)
display(summary)

display(Markdown("### Sample Rows"))
preview_cols = ["provider", "date", "symbol", "open", "high", "low", "close", "volume", "target", "rank_y", "event", "horizon"]
display(pd.concat([provider_datasets[p].head(3) for p in PROVIDERS], ignore_index=True)[preview_cols])

,provider,symbols,rows,train_rows,dev_rows,test_rows,positive_rate,first_label_date,last_label_date,unique_horizons
0,yfinance,7,1405,1011,204,190,0.503915,2020-01-02,2026-06-26,12
1,fmp,7,1403,1012,204,187,0.504633,2020-01-02,2026-06-24,12


### Sample Rows

,provider,date,symbol,open,high,low,close,volume,target,rank_y,event,horizon
0,yfinance,2020-01-06,AAPL,70.754014,72.239942,70.503546,72.201408,118387200.0,1,0.302030,entry,YE_k7
1,yfinance,2020-01-29,AAPL,78.137915,78.956742,77.398559,78.111420,216229200.0,0,0.243655,exit,YE_k12
2,yfinance,2020-02-03,AAPL,73.285151,75.498397,72.784224,74.335182,173788400.0,1,0.106599,entry,YE_k12
3,fmp,2020-01-06,AAPL,70.750000,72.230000,70.490000,72.190000,118578576.0,1,0.305128,entry,YE_k8
4,fmp,2020-01-29,AAPL,78.130000,78.950000,77.400000,78.100000,216599712.0,0,0.246154,exit,YE_k12
5,fmp,2020-02-03,AAPL,73.280000,75.490000,72.780000,74.330000,173985604.0,1,0.102564,entry,YE_k12


## Model Training Helpers

The RandomForest is a scikit-learn style model trained with RAPIDS cuML on CUDA. The autoencoder is a PyTorch CUDA model. The Flair model uses a tiny pretrained transformer and Flair's `MultitaskModel` for classification plus return-percentile regression.

In [5]:
def train_cuml_random_forest(provider: str, splits: dict[str, pd.DataFrame], arrays: dict[str, np.ndarray]) -> dict[str, object]:
    import cudf
    from cuml.ensemble import RandomForestClassifier

    started = perf_counter()
    model = RandomForestClassifier(
        n_estimators=128,
        max_depth=8,
        random_state=RANDOM_SEED,
        n_streams=1,
    )
    x_train = cudf.DataFrame(pd.DataFrame(arrays["train"], columns=OHLCV_COLUMNS))
    y_train = cudf.Series(splits["train"]["target"].astype("int32").reset_index(drop=True))
    model.fit(x_train, y_train)

    x_test = cudf.DataFrame(pd.DataFrame(arrays["test"], columns=OHLCV_COLUMNS))
    y_test = splits["test"]["target"].astype(int).to_numpy()
    pred = model.predict(x_test).to_numpy().astype(int)
    proba_raw = model.predict_proba(x_test)
    proba = proba_raw.to_numpy()[:, 1] if hasattr(proba_raw, "to_numpy") else np.asarray(proba_raw)[:, 1]
    elapsed = perf_counter() - started
    return {
        "provider": provider,
        "framework": "sklearn API via RAPIDS cuML",
        "model": "RandomForestClassifier",
        "device": "cuda",
        "test_accuracy": accuracy_score(y_test, pred),
        "test_f1": f1_score(y_test, pred, zero_division=0),
        "test_roc_auc": roc_auc_score(y_test, proba) if len(np.unique(y_test)) == 2 else np.nan,
        "test_mae": np.nan,
        "train_seconds": elapsed,
    }


class TinyAutoencoder(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int = 3):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, 8), nn.ReLU(), nn.Linear(8, latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 8), nn.ReLU(), nn.Linear(8, input_dim))

    def forward(self, values: torch.Tensor) -> torch.Tensor:
        return self.decoder(self.encoder(values))


def train_torch_autoencoder(provider: str, arrays: dict[str, np.ndarray]) -> dict[str, object]:
    started = perf_counter()
    model = TinyAutoencoder(input_dim=len(OHLCV_COLUMNS)).to(TORCH_DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()
    train_tensor = torch.tensor(arrays["train"], dtype=torch.float32)
    loader = DataLoader(TensorDataset(train_tensor), batch_size=32, shuffle=True)
    for _ in range(TORCH_EPOCHS):
        model.train()
        for (batch,) in loader:
            batch = batch.to(TORCH_DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(model(batch), batch)
            loss.backward()
            optimizer.step()
    model.eval()
    with torch.no_grad():
        train_x = torch.tensor(arrays["train"], dtype=torch.float32, device=TORCH_DEVICE)
        test_x = torch.tensor(arrays["test"], dtype=torch.float32, device=TORCH_DEVICE)
        train_mse = loss_fn(model(train_x), train_x).detach().cpu().item()
        test_mse = loss_fn(model(test_x), test_x).detach().cpu().item()
    elapsed = perf_counter() - started
    return {
        "provider": provider,
        "framework": "PyTorch",
        "model": "TinyAutoencoder",
        "device": str(TORCH_DEVICE),
        "test_accuracy": np.nan,
        "test_f1": np.nan,
        "test_roc_auc": np.nan,
        "test_mae": np.nan,
        "train_reconstruction_mse": train_mse,
        "test_reconstruction_mse": test_mse,
        "train_seconds": elapsed,
    }

In [6]:
def train_flair_multitask(provider: str, splits: dict[str, pd.DataFrame]) -> dict[str, object]:
    started = perf_counter()
    result = train_classification_regression_multitask(
        splits,
        base_path=ARTIFACT_DIR / f"flair_mtl_{provider}",
        transformer_model=FLAIR_TRANSFORMER,
        text_column="text",
        classification_column="target",
        regression_column="rank_y",
        class_label_fn=lambda value: "long" if int(value) == 1 else "not_long",
        classification_label_type="direction",
        regression_label_type="return_percentile",
        classification_task_id="direction",
        regression_task_id="return_percentile",
        classification_loss_factor=1.0,
        regression_loss_factor=0.5,
        fine_tune_transformer=False,
        max_epochs=FLAIR_EPOCHS,
        learning_rate=1e-4,
        mini_batch_size=16,
        mini_batch_chunk_size=8,
        eval_batch_size=32,
        save_final_model=False,
        create_file_logs=False,
        create_loss_file=False,
        embeddings_storage_mode="none",
    )
    y_pred_direction_labels, y_pred_return = predict_classification_regression(
        result,
        result.corpus.test,
        classification_prediction_label="pred_direction",
        regression_prediction_label="pred_return_percentile",
        mini_batch_size=32,
    )
    y_true_direction = splits["test"]["target"].astype(int).to_numpy()
    y_pred_direction = np.array([1 if label == "long" else 0 for label in y_pred_direction_labels])
    y_true_return = splits["test"]["rank_y"].astype(float).to_numpy()
    elapsed = perf_counter() - started
    return {
        "provider": provider,
        "framework": "FlairNLP",
        "model": f"MultitaskModel({FLAIR_TRANSFORMER})",
        "device": str(TORCH_DEVICE),
        "test_accuracy": accuracy_score(y_true_direction, y_pred_direction),
        "test_f1": f1_score(y_true_direction, y_pred_direction, zero_division=0),
        "test_roc_auc": np.nan,
        "test_mae": mean_absolute_error(y_true_return, y_pred_return),
        "train_seconds": elapsed,
    }

In [7]:
results = []
for provider in PROVIDERS:
    display(Markdown(f"### Training models for `{provider}`"))
    splits = provider_splits[provider]
    arrays = provider_arrays[provider]
    for trainer in (train_cuml_random_forest, train_torch_autoencoder, train_flair_multitask):
        if trainer is train_cuml_random_forest:
            row = trainer(provider, splits, arrays)
        elif trainer is train_torch_autoencoder:
            row = trainer(provider, arrays)
        else:
            row = trainer(provider, splits)
        results.append(row)
        display(pd.DataFrame([row]).dropna(axis=1, how="all"))

results_df = pd.DataFrame(results)
metric_cols = [
    "provider",
    "framework",
    "model",
    "device",
    "test_accuracy",
    "test_f1",
    "test_roc_auc",
    "test_mae",
    "train_reconstruction_mse",
    "test_reconstruction_mse",
    "train_seconds",
]
results_df = results_df.reindex(columns=metric_cols)
display(Markdown("## Cross-Provider / Cross-Framework Results"))
display(results_df)

### Training models for `yfinance`

[06:58:24] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_aarch64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.7.0


,provider,framework,model,device,test_accuracy,test_f1,test_roc_auc,train_seconds
0,yfinance,sklearn API via RAPIDS cuML,RandomForestClassifier,cuda,0.442105,0.302632,0.486919,2.310369


,provider,framework,model,device,train_reconstruction_mse,test_reconstruction_mse,train_seconds
0,yfinance,PyTorch,TinyAutoencoder,cuda,0.043987,0.020841,1.347633


2026-06-29 06:58:27,911 Computing label dictionary. Progress:


...

2026-06-29 06:58:32,117 --------------------------------------------------

direction - Label type: direction


Results:
- F-score (micro) 0.4385
- F-score (macro) 0.4169
- Accuracy 0.4385

By class:
              precision    recall  f1-score   support

    not_long     0.4470    0.6484    0.5291        91
        long     0.4182    0.2396    0.3046        96

    accuracy                         0.4385       187
   macro avg     0.4326    0.4440    0.4169       187
weighted avg     0.4322    0.4385    0.4139       187
--------------------------------------------------

return_percentile - Label type: return_percentile

AVG: mse: 1.0194 - mae: 0.9517 - pearson: -0.1464 - spearman: -0.1253


2026-06-29 06:58:32,118 ----------------------------------------------------------------------------------------------------


,provider,framework,model,device,test_accuracy,test_f1,test_mae,train_seconds
0,fmp,FlairNLP,MultitaskModel(prajjwal1/bert-tiny),cuda,0.438503,0.304636,0.951664,2.163644


## Cross-Provider / Cross-Framework Results

,provider,framework,model,device,test_accuracy,test_f1,test_roc_auc,test_mae,train_reconstruction_mse,test_reconstruction_mse,train_seconds
0,yfinance,sklearn API via RAPIDS cuML,RandomForestClassifier,cuda,0.442105,0.302632,0.486919,NaN,NaN,NaN,2.310369
1,yfinance,PyTorch,TinyAutoencoder,cuda,NaN,NaN,NaN,NaN,0.043987,0.020841,1.347633
2,yfinance,FlairNLP,MultitaskModel(prajjwal1/bert-tiny),cuda,0.489474,0.126126,NaN,0.271794,NaN,NaN,2.489784
3,fmp,sklearn API via RAPIDS cuML,RandomForestClassifier,cuda,0.524064,0.410596,0.554258,NaN,NaN,NaN,0.312385
4,fmp,PyTorch,TinyAutoencoder,cuda,NaN,NaN,NaN,NaN,0.020801,0.034507,0.816645
5,fmp,FlairNLP,MultitaskModel(prajjwal1/bert-tiny),cuda,0.438503,0.304636,NaN,0.951664,NaN,NaN,2.163644


In [8]:
readable = results_df.copy()
for col in ["test_accuracy", "test_f1", "test_roc_auc", "test_mae", "train_reconstruction_mse", "test_reconstruction_mse", "train_seconds"]:
    readable[col] = pd.to_numeric(readable[col], errors="coerce").round(4)

rf_rows = readable[readable["framework"].str.contains("cuML", na=False)].sort_values("test_f1", ascending=False)
flair_rows = readable[readable["framework"].eq("FlairNLP")].sort_values("test_f1", ascending=False)
torch_rows = readable[readable["framework"].eq("PyTorch")].sort_values("test_reconstruction_mse", ascending=True)

lines = [
    "## Notebook Readout",
    f"- Quant Warehouse produced {int(summary['rows'].sum())} labeled rows across {len(PROVIDERS)} data vendors and {len(SYMBOLS)} symbols.",
]
if not rf_rows.empty:
    best = rf_rows.iloc[0]
    lines.append(
        f"- Best CUDA RandomForest classification run: {best['provider']} with F1={best['test_f1']} and ROC AUC={best['test_roc_auc']}."
    )
if not flair_rows.empty:
    best = flair_rows.iloc[0]
    lines.append(
        f"- Best tiny-transformer Flair direction run: {best['provider']} with F1={best['test_f1']} and return-percentile MAE={best['test_mae']}."
    )
if not torch_rows.empty:
    best = torch_rows.iloc[0]
    lines.append(
        f"- Best PyTorch autoencoder reconstruction run: {best['provider']} with test MSE={best['test_reconstruction_mse']}."
    )
lines.append(
    "- This is intentionally a toy training notebook: it validates data movement, CUDA selection, framework differences, and target reuse before platform abstractions are tightened."
)
display(Markdown("\n".join(lines)))
display(readable)

## Notebook Readout
- Quant Warehouse produced 2808 labeled rows across 2 data vendors and 7 symbols.
- Best CUDA RandomForest classification run: fmp with F1=0.4106 and ROC AUC=0.5543.
- Best tiny-transformer Flair direction run: fmp with F1=0.3046 and return-percentile MAE=0.9517.
- Best PyTorch autoencoder reconstruction run: yfinance with test MSE=0.0208.
- This is intentionally a toy training notebook: it validates data movement, CUDA selection, framework differences, and target reuse before platform abstractions are tightened.

,provider,framework,model,device,test_accuracy,test_f1,test_roc_auc,test_mae,train_reconstruction_mse,test_reconstruction_mse,train_seconds
0,yfinance,sklearn API via RAPIDS cuML,RandomForestClassifier,cuda,0.4421,0.3026,0.4869,NaN,NaN,NaN,2.3104
1,yfinance,PyTorch,TinyAutoencoder,cuda,NaN,NaN,NaN,NaN,0.0440,0.0208,1.3476
2,yfinance,FlairNLP,MultitaskModel(prajjwal1/bert-tiny),cuda,0.4895,0.1261,NaN,0.2718,NaN,NaN,2.4898
3,fmp,sklearn API via RAPIDS cuML,RandomForestClassifier,cuda,0.5241,0.4106,0.5543,NaN,NaN,NaN,0.3124
4,fmp,PyTorch,TinyAutoencoder,cuda,NaN,NaN,NaN,NaN,0.0208,0.0345,0.8166
5,fmp,FlairNLP,MultitaskModel(prajjwal1/bert-tiny),cuda,0.4385,0.3046,NaN,0.9517,NaN,NaN,2.1636
